# ScholarAgent error analysis (Phase 8)

Load a saved evaluation run and inspect failures by system and question type.
Default paths assume you ran:

```bash
uv run scholar-agent evaluate --max-questions 50 --embedding-backend hash
```

In [ ]:
from pathlib import Path
import csv
import json
from collections import defaultdict

OUT = Path("outputs/evaluation")
results_path = OUT / "results.json"
fail_path = OUT / "failures.json"
pq_path = OUT / "per_question_metrics.csv"

report = json.loads(results_path.read_text()) if results_path.is_file() else {}
failures = json.loads(fail_path.read_text()) if fail_path.is_file() else []
print("run_id:", report.get("run_id"))
print("fingerprint:", report.get("frozen_split_fingerprint"))
print("n failures:", len(failures))
groups = defaultdict(list)
if pq_path.is_file():
    with pq_path.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            groups[(row["system"], row["question_type"])].append(row)
    for key, rows in sorted(groups.items()):
        means = {name: sum(float(row[name]) for row in rows) / len(rows) for name in ("recall_at_k_paper", "token_f1", "latency_ms")}
        print(key, means)
else:
    print("per_question_metrics.csv missing — run evaluate first")

## Manual review checklist (≥5 cases)

1. `q_k08`: dense miss vs hybrid exact terminology
2. `q_r09`: graph recall gain and latency cost
3. `q_c15`: graph slot displacement regression
4. `q_c05`: planner comparison coverage gain
5. `q_u01`: full-agent refusal failure
6. `q_u03`: scope evidence misread as answer evidence

See `docs/failure_analysis.md` for the written narratives.

In [ ]:
from collections import Counter
print(Counter((f.get("system"), f.get("question_type")) for f in failures).most_common(20))
for f in failures[:10]:
    print("-" * 60)
    print(f.get("system"), f.get("question_id"), f.get("question_type"))
    print(f.get("question"))
    print("reason:", f.get("reason"))
    print("preview:", (f.get("answer_preview") or "")[:200])